# Module 5: k-Shingling, MinHash Signatures & Locality-Sensitive Hashing

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Universal hash signatures, Jaccard estimation, LSH banding, and S-curve verification.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM05_LSH_MinHash_Similarity.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. $k$-Shingling, Exact Jaccard Similarity & MinHash Signatures

Exact all-pairs comparison over $N$ documents scales as $O(N^2)$. MinHash compresses each shingle set into a short integer signature whose component-wise agreement estimates Jaccard similarity.

In [1]:
import hashlib
import random


def k_shingles(text, k=3):
    clean = " ".join(text.lower().split())
    return {clean[i : i + k] for i in range(len(clean) - k + 1)}


def jaccard(a, b):
    return len(a & b) / len(a | b)


def minhash_signature(shingles, coeffs, prime=(1 << 61) - 1):
    ids = [int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(), "big") for s in shingles]
    return [min((a * x + b) % prime for x in ids) for a, b in coeffs]


prime = (1 << 61) - 1
rng = random.Random(3003)
coeffs = [(rng.randrange(1, prime), rng.randrange(0, prime)) for _ in range(100)]

d1 = k_shingles("zanzibar clove harvest in pemba north during masika rains")
d2 = k_shingles("zanzibar clove harvest in pemba south during masika rains")
d3 = k_shingles("distributed mapreduce word count with apache spark")

sig1, sig2, sig3 = (minhash_signature(d, coeffs, prime) for d in (d1, d2, d3))
est12 = sum(x == y for x, y in zip(sig1, sig2)) / 100
est13 = sum(x == y for x, y in zip(sig1, sig3)) / 100
print(f"d1 vs d2 -> exact Jaccard={jaccard(d1, d2):.3f}, MinHash={est12:.3f}")
print(f"d1 vs d3 -> exact Jaccard={jaccard(d1, d3):.3f}, MinHash={est13:.3f}")
assert abs(est12 - jaccard(d1, d2)) < 0.12


d1 vs d2 -> exact Jaccard=0.833, MinHash=0.830
d1 vs d3 -> exact Jaccard=0.020, MinHash=0.010


## 2. LSH Banding & the S-Curve Threshold

Splitting a 100-row signature into $b = 20$ bands of $r = 5$ rows yields candidate probability $P(\text{match}) = 1 - (1 - s^r)^b$, creating a sharp threshold near $(1/b)^{1/r} \approx 0.55$.

In [2]:
docs = {"d1": sig1, "d2": sig2, "d3": sig3, "d4": minhash_signature(k_shingles("distributed mapreduce log analysis with apache spark"), coeffs, prime)}
b, r = 20, 5
candidates = set()
for band in range(b):
    buckets = {}
    for name, sig in docs.items():
        key = tuple(sig[band * r : (band + 1) * r])
        buckets.setdefault(key, []).append(name)
    for members in buckets.values():
        if len(members) > 1:
            members.sort()
            for i in range(len(members)):
                for j in range(i + 1, len(members)):
                    candidates.add((members[i], members[j]))

print("LSH candidate pairs:", sorted(candidates))
for s in (0.1, 0.3, 0.55, 0.8):
    print(f"  s={s:.2f} -> P(candidate) = {1 - (1 - s**r)**b:.4f}")
assert ("d1", "d2") in candidates and ("d1", "d3") not in candidates


LSH candidate pairs: [('d1', 'd2')]
  s=0.10 -> P(candidate) = 0.0002
  s=0.30 -> P(candidate) = 0.0475
  s=0.55 -> P(candidate) = 0.6440
  s=0.80 -> P(candidate) = 0.9996
